In [178]:
import json
from pathlib import Path

import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [179]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "training_data.json"
MODELS_DIR = BASE_DIR / "models"

MODELS_DIR.mkdir(exist_ok=True)

In [180]:
CATEGORY_MERGE_MAP = {
    "Education/Homework": "Education/Assignments",
    "Education/Labworks": "Education/Assignments",

    "Education/Certificates": "Documents/Personal",
    "Personal/Documents": "Documents/Personal",

    "Media/Books": "Education/Materials",
    "Education/Materials": "Education/Materials",

    "Career/Internship": "Career/Internship",
    "Career/Resume": "Career/Resume",
    "Data/Datasets": "Education/Materials",
    "Education/Project": "Education/Project",
    "Finance": "Finance",
    "IT/Licenses": "IT/Licenses",
    "Media/Transcript": "Media/Transcript",
}


def merge_category(category: str) -> str:
    return CATEGORY_MERGE_MAP.get(category, category)

In [181]:
texts = []
labels = []

with open(DATA_PATH) as f:
    jf = json.load(f)

for note in jf:
    texts.append(note["text"])
    labels.append(note["category"])

In [182]:
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [183]:
pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 3),
            min_df=1,
            sublinear_tf=True,
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=1,
            sublinear_tf=True,
        )),
    ])),
    ("clf", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        C=1.5,
        solver="lbfgs",
    ))
])

In [184]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('features', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('word_tfidf', ...), ('char_tfidf', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'


In [185]:
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.9320388349514563

                        precision    recall  f1-score   support

     Career/Internship       0.94      0.94      0.94        17
         Career/Resume       0.93      0.87      0.90        15
         Data/Datasets       0.95      0.95      0.95        21
Education/Certificates       0.82      1.00      0.90         9
    Education/Homework       0.83      0.83      0.83        12
    Education/Labworks       1.00      0.92      0.96        26
   Education/Materials       0.88      0.94      0.91        16
     Education/Project       1.00      1.00      1.00         9
               Finance       0.91      0.91      0.91        23
           IT/Licenses       0.93      1.00      0.96        13
           Media/Books       1.00      0.80      0.89        15
      Media/Transcript       0.91      1.00      0.95        21
    Personal/Documents       1.00      1.00      1.00         9

              accuracy                           0.93       206
        

In [177]:
MODEL_PATH = MODELS_DIR / "classifier.joblib"

joblib.dump(pipeline, MODEL_PATH)

print(f"Модель сохранена в {MODEL_PATH}")

Модель сохранена в /home/sarumo/Documents/Projects/TidyFS/classifier/models/classifier.joblib
